# Barcelona Noise Prediction: OSM Building Heights
This notebook downloads the building footprints from OpenStreetMap (OSM), processes their heights (falling back to number of levels * 3m where precise height is missing), and evaluates the average and max building height within 50m and 100m buffers around the streets.

In [ ]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
import os
import numpy as np

# Try to set osmnx to use overpass efficiently
ox.settings.timeout = 1000

print("Loading street data...")
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")
print("Streets CRS:", noise_streets.crs)
print("Number of street segments:", len(noise_streets))

## Download Building Data from OSM
We use `osmnx` to fetch building polygons for Barcelona. Due to the high number of buildings, this can take a few minutes.

In [ ]:
place_name = "Barcelona, Spain"
tags = {'building': True}

print("Downloading buildings from OSM...")
# Note: depending on the osmnx version, this function could be ox.geometries_from_place or ox.features_from_place
try:
    buildings = ox.features_from_place(place_name, tags)
except AttributeError:
    buildings = ox.geometries_from_place(place_name, tags)

print(f"Downloaded {len(buildings)} buildings.")
# Ensure consistent geometry type and CRS
buildings = buildings[buildings.geometry.type.isin(['Polygon', 'MultiPolygon'])]
buildings = buildings.to_crs(noise_streets.crs)
display(buildings.head(3))

## Process Building Heights
OSM doesn't always have exact height in meters ('height' tag).
We'll try to extract 'height' first. If that fails, we fallback to 'building:levels' multiplying each level by 3 meters.
If neither is present, we impute the height using the median building height of the city.

In [ ]:
def clean_height_col(series):
    # Remove non-numeric characters (like 'm', spaces) and convert to float
    if series.dtype == 'O':
        return pd.to_numeric(series.str.replace(r'[^\d.]', '', regex=True), errors='coerce')
    return pd.to_numeric(series, errors='coerce')

# Initialize a series for estimated height
estimated_heights = pd.Series(index=buildings.index, dtype=float)

# 1. Extract exact height if available
if 'height' in buildings.columns:
    estimated_heights = clean_height_col(buildings['height'])
    print(f"Valid exact heights found: {estimated_heights.notna().sum()}")

# 2. Extract levels and multiply by 3
if 'building:levels' in buildings.columns:
    levels = clean_height_col(buildings['building:levels'])
    height_from_levels = levels * 3.0
    # Fill missing exact heights with height from levels
    estimated_heights = estimated_heights.fillna(height_from_levels)
    print(f"Calculated from levels: {height_from_levels.notna().sum()}")

# 3. Give remaining buildings the median height (or a fallback like 6m / 2 stories for a city)
median_height = estimated_heights.median()
if np.isnan(median_height):
    median_height = 9.0  # Fallback to 3 stories if dataset is too sparse
    
estimated_heights = estimated_heights.fillna(median_height)
buildings['estimated_height'] = estimated_heights

print(f"All buildings have an assigned height. Median is: {median_height:.1f}m")

fig, ax = plt.subplots(figsize=(6, 4))
buildings['estimated_height'].plot(kind='hist', bins=50, ax=ax, color='skyblue', edgecolor='black')
ax.set_title('Distribution of Estimated Building Heights')
ax.set_xlabel('Height in meters')
ax.set_xlim(0, 50)  # Focus on the majority
plt.show()

## Buffer Analysis
We calculate the mean and max building height within 50m and 100m for each street. Because we evaluate intersecting buildings, we'll perform a Spatial Join (sjoin).

In [ ]:
def calculate_building_heights_in_buffer(streets_gdf, buildings_gdf, buffer_size):
    print(f"Evaluating {buffer_size}m buffers...")
    # Buffer the streets
    buffered_streets = streets_gdf.copy()
    buffered_streets['geometry'] = buffered_streets.geometry.buffer(buffer_size)
    
    # Keep only target columns for spatial join to reduce memory usage
    bldgs = buildings_gdf[['estimated_height', 'geometry']].copy()
    
    # Spatial join: Which buildings intersect which street buffer?
    joined = gpd.sjoin(buffered_streets[['TRAM', 'geometry']], bldgs, how='left', predicate='intersects')
    
    # If a street buffer hits nothing, it will have NaN in estimated_height. We can treat that as 0 (no buildings).
    joined['estimated_height'] = joined['estimated_height'].fillna(0)
    
    # Group by street and calculate aggregate stats
    agg_funcs = {
        'estimated_height': ['mean', 'max']
    }
    grouped = joined.groupby('TRAM').agg(agg_funcs)
    
    # Flatten MultiIndex columns
    grouped.columns = [f"bldg_h_{col[1]}_{buffer_size}m" for col in grouped.columns]
    
    return grouped.reset_index()

features_50m = calculate_building_heights_in_buffer(noise_streets, buildings, 50)
features_100m = calculate_building_heights_in_buffer(noise_streets, buildings, 100)

display(features_50m.head())

## Merge and Export

In [ ]:
dataset = pd.DataFrame({'street_id': noise_streets['TRAM']})

dataset = dataset.merge(features_50m, left_on='street_id', right_on='TRAM', how='left').drop(columns=['TRAM'])
dataset = dataset.merge(features_100m, left_on='street_id', right_on='TRAM', how='left').drop(columns=['TRAM'])

dataset = dataset.fillna(0)
display(dataset.head())

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
dataset.to_csv(os.path.join(output_dir, "osm_building_heights.csv"), index=False)
print("Exported osm_building_heights.csv")